# Logistic Regression for Classification Task
### A practice for Week 4 - DataLab I

## What is the difference between Linear and Logistic Regression?
Despite the fact that linear regression is effective at predicting continuous values (e.g. Co2 emissions in cars), it is not the best technique for predicting the class of an observed data point. For the purpose of estimating a data point's class, we need some guidance on what class would be most likely. Logistic Regression is used for this purpose.



<b>Logistic Regression</b> is a variation of Linear Regression, useful when the target variable <b>(y)</b>, is categorical. It produces a formula that predicts the probability of the class label as a function of the independent variables.

In this practice, you will learn Logistic Regression with creating a model for a telecommunication company, to predict when its customers will leave for a competitor, so that they can take some action to retain the customers.

In [29]:
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pylab as pl
import scipy.optimize as opt
%matplotlib inline 
import matplotlib.pyplot as plt

<h2 id="about_dataset">About the dataset</h2>
We will use a telecommunications dataset for predicting customer churn. This is a historical customer dataset where each row represents one customer. The focus of this analysis is to predict the customers who will stay with the company. 

This data set provides information to help you predict what behavior will help you to retain customers. You can analyze all relevant customer data and develop focused customer retention programs.

The dataset includes information about:

*   Customers who left within the last month – the column is called Churn
*   Services that each customer has signed up for – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies
*   Customer account information – how long they had been a customer, contract, payment method, paperless billing, monthly charges, and total charges
*   Demographic info about customers – gender, age range, and if they have partners and dependents

## Loading the data

In [3]:
df = pd.read_csv("CustomerData.csv")
df.head()

,tenure,age,address,income,ed,employ,equip,callcard,wireless,longmon,...,pager,internet,callwait,confer,ebill,loglong,logtoll,lninc,custcat,churn
0,11.0,33.0,7.0,136.0,5.0,5.0,0.0,1.0,1.0,4.40,...,1.0,0.0,1.0,1.0,0.0,1.482,3.033,4.913,4.0,1.0
1,33.0,33.0,12.0,33.0,2.0,0.0,0.0,0.0,0.0,9.45,...,0.0,0.0,0.0,0.0,0.0,2.246,3.240,3.497,1.0,1.0
2,23.0,30.0,9.0,30.0,1.0,2.0,0.0,0.0,0.0,6.30,...,0.0,0.0,0.0,1.0,0.0,1.841,3.240,3.401,3.0,0.0
3,38.0,35.0,5.0,76.0,2.0,10.0,1.0,1.0,1.0,6.05,...,1.0,1.0,1.0,1.0,1.0,1.800,3.807,4.331,4.0,0.0
4,7.0,35.0,14.0,80.0,2.0,15.0,0.0,1.0,0.0,7.10,...,0.0,0.0,1.0,1.0,0.0,1.960,3.091,4.382,3.0,0.0


<h2 id="preprocessing">Data preprocessing and selection</h2>

Select some correlated features for the modeling. Also, we change the target data type to be an integer, as it is a requirement by algorithm:

In [4]:
df = df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip',   'callcard', 'wireless','churn']]
df['churn'] = df['churn'].astype('int')
df.head()

,tenure,age,address,income,ed,employ,equip,callcard,wireless,churn
0,11.0,33.0,7.0,136.0,5.0,5.0,0.0,1.0,1.0,1
1,33.0,33.0,12.0,33.0,2.0,0.0,0.0,0.0,0.0,1
2,23.0,30.0,9.0,30.0,1.0,2.0,0.0,0.0,0.0,0
3,38.0,35.0,5.0,76.0,2.0,10.0,1.0,1.0,1.0,0
4,7.0,35.0,14.0,80.0,2.0,15.0,0.0,1.0,0.0,0


### What's the dataset shape, info and description?

In [ ]:
# write your code here as a practice

<details><summary>Click here for the solution</summary>

```python
df.shape
df.info()
df.describe()
```
</details>

Let's define independent variables (X), and target variable (y) for our dataset:

In [8]:
X = np.asarray(df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip']])
X[0:10]

array([[ 11.,  33.,   7., 136.,   5.,   5.,   0.],
       [ 33.,  33.,  12.,  33.,   2.,   0.,   0.],
       [ 23.,  30.,   9.,  30.,   1.,   2.,   0.],
       [ 38.,  35.,   5.,  76.,   2.,  10.,   1.],
       [  7.,  35.,  14.,  80.,   2.,  15.,   0.],
       [ 68.,  52.,  17., 120.,   1.,  24.,   0.],
       [ 42.,  40.,   7.,  37.,   2.,   8.,   1.],
       [  9.,  21.,   1.,  17.,   2.,   2.,   0.],
       [ 35.,  50.,  26., 140.,   2.,  21.,   0.],
       [ 49.,  51.,  27.,  63.,   4.,  19.,   0.]])

In [9]:
y = np.asarray(df['churn'])
y [0:10]

array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0])

## Splitting for Train and Test dataset

In [43]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

Train set: (160, 7) (160,)
Test set: (40, 7) (40,)


<h2 id="modeling">Modeling (Logistic Regression with Scikit-learn)</h2>

Let's build our model using **LogisticRegression** from the Scikit-learn package in Python. This function implements logistic regression.

The version of Logistic Regression in Scikit-learn, support regularization. Regularization is a technique used to solve the overfitting problem of machine learning models.
**C** parameter indicates **inverse of regularization strength** which must be a positive float. Smaller values specify stronger regularization.
Now let's fit our model with train set:

In [44]:
from sklearn.linear_model import LogisticRegression
LR = LogisticRegression()
LR.fit(X_train, y_train)

LogisticRegression()

Now we can predict using our test set:

In [45]:
y_predict = LR.predict(X_test)
y_predict

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0])

**predict_probability**  returns estimates for all classes, ordered by the label of classes. So, the first column is the probability of class 0, P(Y=0|X), and second column is probability of class 1, P(Y=1|X):

In [46]:
y_predict_probability = LR.predict_proba(X_test)
y_predict_probability

array([[0.53, 0.47],
       [0.2 , 0.8 ],
       [0.78, 0.22],
       [0.68, 0.32],
       [0.83, 0.17],
       [0.3 , 0.7 ],
       [0.98, 0.02],
       [0.54, 0.46],
       [0.53, 0.47],
       [0.89, 0.11],
       [0.91, 0.09],
       [0.94, 0.06],
       [0.82, 0.18],
       [0.97, 0.03],
       [0.42, 0.58],
       [0.91, 0.09],
       [0.65, 0.35],
       [0.48, 0.52],
       [0.75, 0.25],
       [0.61, 0.39],
       [0.95, 0.05],
       [0.63, 0.37],
       [0.31, 0.69],
       [0.49, 0.51],
       [0.79, 0.21],
       [0.96, 0.04],
       [0.49, 0.51],
       [0.93, 0.07],
       [0.93, 0.07],
       [0.84, 0.16],
       [0.9 , 0.1 ],
       [0.67, 0.33],
       [0.51, 0.49],
       [0.85, 0.15],
       [0.74, 0.26],
       [0.37, 0.63],
       [0.95, 0.05],
       [0.68, 0.32],
       [0.95, 0.05],
       [0.75, 0.25]])

In [47]:
accuracy = accuracy_score(y_test, y_predict)
precision = precision_score(y_test, y_predict)
recall = recall_score(y_test, y_predict)
f1 = f1_score(y_test, y_predict)
report = classification_report(y_test, y_predict)

Based on the count of each section, we can calculate precision and recall of each label:

*   **Precision** is a measure of the accuracy provided that a class label has been predicted. It is defined by: precision = TP / (TP + FP)

*   **Recall** is the true positive rate. It is defined as: Recall =  TP / (TP + FN)

So, we can calculate the precision and recall of each class.



In [49]:
evaluation_results = {
    
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Accuracy": accuracy
}
print (evaluation_results)
evaluation_results, report


{'Precision': 0.5, 'Recall': 0.4444444444444444, 'F1-Score': 0.47058823529411764, 'Accuracy': 0.775}


({'Precision': 0.5,
  'Recall': 0.4444444444444444,
  'F1-Score': 0.47058823529411764,
  'Accuracy': 0.775},
 '              precision    recall  f1-score   support\n\n           0       0.84      0.87      0.86        31\n           1       0.50      0.44      0.47         9\n\n    accuracy                           0.78        40\n   macro avg       0.67      0.66      0.66        40\nweighted avg       0.77      0.78      0.77        40\n')

**F1 score:**
Now we are in the position to calculate the F1 scores for each label based on the precision and recall of that label.

The F1 score is the harmonic average of the precision and recall, where an F1 score reaches its best value at 1 (perfect precision and recall) and worst at 0. It is a good way to show that a classifer has a good value for both recall and precision.

Finally, we can tell the average accuracy for this classifier is the average of the F1-score for both labels, which is 0.77 in our case.

### Evaluation Metrics:

Predictions were evaluated using accuracy, precision, recall, and F1-score.

- **Accuracy:** Proportion of correctly classified instances.
- **Precision:** Proportion of positive predictions that were actually correct.
- **Recall:** Proportion of actual positives correctly identified.
- **F1-Score:** Harmonic mean of precision and recall.

**Classification Report:**

Class 0 (non-churn) is better predicted with higher with higher precision (84%) and recall (87%) compared to class 1 (non-churn) with precision (50%) and recall (44%).

The classifier correctly predicted 31 of them as 0. So, it has done a good job in predicting the customers with churn value 0. In a specific case of the binary classifier, such as this example, we can interpret true predicted and false predicted numbers as the count of true positives, false positives, true negatives, and false negatives.

